In [ ]:
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt

# Define um estilo visual mais agradável para os gráficos
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported and style set.")

In [ ]:
# Célula 2 (Com Ordenação Natural)

import pandas as pd
import os
import glob

results_dir = 'results'
experiment_data = []

stat_files = glob.glob(os.path.join(results_dir, '**', '*_stats.txt'), recursive=True)

for file_path in stat_files:
    data = {}
    with open(file_path, 'r') as f:
        next(f, None) # Pula a primeira linha "--- ..."
        for line in f:
            if ':' in line:
                key, value = line.split(':', 1)
                data[key.strip()] = value.strip()
    
    instance_info = os.path.basename(os.path.dirname(file_path))
    data['instance_base'] = instance_info.split('_lambda_')[0]
    
    experiment_data.append(data)

df_results = pd.DataFrame(experiment_data)

# Converte as colunas para o tipo numérico usando as chaves corretas
df_results['objective_value'] = pd.to_numeric(df_results['Objective_Value'])
df_results['num_clusters'] = pd.to_numeric(df_results['Number_of_Clusters'])
df_results['execution_time_s'] = pd.to_numeric(df_results['Solver_Execution_Time_s'])
df_results['lambda'] = pd.to_numeric(df_results['Lambda Weight'])

# --- CORREÇÃO DA ORDENAÇÃO ---
# 1. Extrai o número da instância (ex: 1, 2, ..., 10) para uma nova coluna
df_results['instance_num'] = df_results['instance_base'].str.extract(r'P(\d+)').astype(int)

# 2. Ordena primeiro pelo número da instância, depois pelo lambda
df_results.sort_values(by=['instance_num', 'lambda'], inplace=True)

# 3. (Opcional) Remove a coluna temporária após a ordenação
df_results.drop(columns=['instance_num'], inplace=True)

print(f"Loaded and processed {len(df_results)} experiment results successfully.")

# Agora o .head() mostrará os resultados do P1, como esperado
display(df_results[['instance_base', 'lambda', 'objective_value', 'num_clusters', 'execution_time_s']].head())

In [ ]:
# Selecione a instância que deseja analisar (ex: a primeira da lista)
# Garante que há resultados antes de tentar plotar
if not df_results.empty:
    instance_to_plot = df_results['instance_base'].unique()[0]
    df_instance = df_results[df_results['instance_base'] == instance_to_plot].copy()

    print(f"Generating Pareto front for instance: {instance_to_plot}")
    print("-" * 50)

    # --- Plot da Fronteira de Pareto ---
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.plot(df_instance['num_clusters'], df_instance['objective_value'], marker='o', 
            markersize=10, linestyle='--', color='b', label='Soluções de Trade-off')

    for i, row in df_instance.iterrows():
        ax.text(row['num_clusters'] + 0.1, row['objective_value'], f" λ={row['lambda']}", 
                fontsize=12, verticalalignment='bottom')

    ax.set_xlabel('Número de Clusters (k) - (Simplicidade)', fontsize=14)
    ax.set_ylabel('Desequilíbrio Esperado - (Qualidade)', fontsize=14)
    ax.set_title(f'Fronteira de Pareto para a Instância: {instance_to_plot}', fontsize=16, pad=20)
    ax.legend(fontsize=12)
    ax.grid(True)
    
    output_fig_path = os.path.join('results', f'pareto_{instance_to_plot}.png')
    plt.savefig(output_fig_path, dpi=300)
    print(f"Pareto front plot saved to '{output_fig_path}'")
    plt.show()

    # --- Análise das Soluções de Destaque ---
    print("\n--- Solução com Foco em SIMPLICIDADE (lambda=0.0) ---")
    display(df_instance[df_instance['lambda'] == 0.0][['num_clusters', 'objective_value']])
    
    print("\n--- Solução com Foco em QUALIDADE (lambda=1.0) ---")
    display(df_instance[df_instance['lambda'] == 1.0][['num_clusters', 'objective_value']])

    print("\n--- Solução de COMPROMISSO (lambda=0.5) ---")
    display(df_instance[df_instance['lambda'] == 0.5][['num_clusters', 'objective_value']])
    print("-" * 50)
else:
    print("DataFrame de resultados está vazio. Execute a Célula 2 primeiro.")

In [ ]:
# --- English ---
# This cell analyzes the results for a specific instance, highlighting the key
# solutions from the Pareto front: the one focused on simplicity (min clusters),
# the one focused on quality (min disagreement), and a balanced trade-off.

# --- Português ---
# Esta célula analisa os resultados para uma instância específica, destacando as 
# soluções-chave da Fronteira de Pareto: a focada em simplicidade (mínimo de clusters),
# a focada em qualidade (mínimo desequilíbrio) e uma solução de compromisso.


# Check if the df_results DataFrame exists and is not empty
if 'df_results' in locals() and not df_results.empty:
    
    # Select the same instance we plotted in the previous cell for consistency
    instance_to_plot = df_results['instance_base'].unique()[0]
    df_instance = df_results[df_results['instance_base'] == instance_to_plot].copy()

    print(f"--- Highlighting Key Solutions for Instance: {instance_to_plot} ---")

    # Solution with focus on SIMPLICITY (minimum number of clusters, lambda=0.0)
    min_k_solution = df_instance[df_instance['lambda'] == 0.0]
    print("\n[+] Solution focused on SIMPLICITY (lambda=0.0):")
    if not min_k_solution.empty:
        display(min_k_solution[['num_clusters', 'objective_value', 'execution_time_s']])
    else:
        print("No solution found for lambda = 0.0")
    print("-" * 60)


    # Solution with focus on QUALITY (minimum disagreement, lambda=1.0)
    min_disagreement_solution = df_instance[df_instance['lambda'] == 1.0]
    print("\n[+] Solution focused on QUALITY (lambda=1.0):")
    if not min_disagreement_solution.empty:
        display(min_disagreement_solution[['num_clusters', 'objective_value', 'execution_time_s']])
    else:
        print("No solution found for lambda = 1.0")
    print("-" * 60)

    # A balanced/trade-off solution (lambda=0.5)
    balanced_solution = df_instance[df_instance['lambda'] == 0.5]
    print("\n[+] BALANCED Trade-off Solution (lambda=0.5):")
    if not balanced_solution.empty:
        display(balanced_solution[['num_clusters', 'objective_value', 'execution_time_s']])
    else:
        print("No solution found for lambda = 0.5")
    print("-" * 60)

else:
    print("ERROR: The 'df_results' DataFrame was not found or is empty.")
    print("Please run the previous cells successfully before this one.")
    print("SUGGESTION: Use the menu 'Kernel' -> 'Restart & Run All' to ensure correct execution order.")